# Tree models versus tabular foundation models

Compare one completed tree CV experiment with a TabPFN-3.5/Causilo run at the same realised-depeg threshold and false-alert budget. The notebook checks fold boundaries and policy settings before comparing the saved out-of-sample summaries. Outer test folds overlap by design; confidence intervals are event/calendar-block bootstrap intervals from each run, not independent-fold intervals or paired significance tests. The validation false-alert budget is not a guarantee on each test fold.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'cv_model_comparison.py').is_file():
    PROJECT_DIR = PROJECT_DIR / '8. Early-Warning Classifier Cross Validation'
if not (PROJECT_DIR / 'cv_model_comparison.py').is_file():
    raise FileNotFoundError('Run from the repository root or folder 8.')
LOG_DIR = PROJECT_DIR / 'lightning_logs'
TREE_EXPERIMENT = 'cv_model_comparison_2026-09-23_15bp'
FOUNDATION_EXPERIMENT = None  # Set a run name, or use the latest foundation run.
DEPEG_THRESHOLD_BPS = 15
FALSE_ALERT_BUDGET = 2.0
if FOUNDATION_EXPERIMENT is None:
    candidates = sorted(LOG_DIR.glob('cv_foundation_models_*_15bp'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError('No foundation CV run yet. Run bash compare_foundation_model_cv_15bp.sh first.')
    FOUNDATION_EXPERIMENT = candidates[-1].name
TREE_DIR = LOG_DIR / TREE_EXPERIMENT
FOUNDATION_DIR = LOG_DIR / FOUNDATION_EXPERIMENT
OUTPUT_DIR = PROJECT_DIR / 'foundation_vs_tree_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
print('Tree:', TREE_DIR)
print('Foundation:', FOUNDATION_DIR)

In [ ]:
def load_summaries(experiment_dir, source):
    paths = list(experiment_dir.glob('*_experiment_summary*/artifacts/comparison/model_comparison_summary.csv'))
    if not paths:
        raise FileNotFoundError(f'No model comparison summaries in {experiment_dir}')
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame['source'] = source
        frame['summary_path'] = str(path)
        frame['source_mtime'] = path.stat().st_mtime
        frames.append(frame)
    result = pd.concat(frames, ignore_index=True)
    required = {'model_name', 'alpha', 'target_threshold', 'false_alert_budget_per_month',
                'cv_event_utility_score_mean', 'cv_timely_event_recall_mean',
                'cv_false_alerts_per_month_mean', 'cv_median_lead_hours_mean',
                'cv_auc_mean', 'cv_auprc_mean', 'cv_brier_score_mean',
                'cv_brier_skill_score_mean'}
    missing = required.difference(result.columns)
    if missing:
        raise ValueError(f'{source} summaries lack columns: {sorted(missing)}')
    result = result.loc[(pd.to_numeric(result['target_threshold']) == DEPEG_THRESHOLD_BPS) &
                        np.isclose(pd.to_numeric(result['false_alert_budget_per_month']), FALSE_ALERT_BUDGET)]
    result = (result.sort_values('source_mtime')
                    .drop_duplicates(['source', 'model_name', 'alpha', 'target_threshold'], keep='last')
                    .reset_index(drop=True))
    if result.empty:
        raise ValueError(f'No {DEPEG_THRESHOLD_BPS}-bp, budget-{FALSE_ALERT_BUDGET} rows in {experiment_dir}')
    return result

summary = pd.concat([load_summaries(TREE_DIR, 'Tree'),
                     load_summaries(FOUNDATION_DIR, 'Foundation')], ignore_index=True)
summary['alpha'] = pd.to_numeric(summary['alpha'])
summary['model_label'] = summary['model_name'].replace({
    'tabpfn_3_5': 'TabPFN 3.5', 'causilo': 'Causilo',
    'random_forest': 'Random forest', 'catboost': 'CatBoost',
    'lightgbm': 'LightGBM', 'xgboost': 'XGBoost',
})
display(summary.groupby('source')['model_name'].agg(['nunique', 'count']))

## Comparability audit

The same dataset and fold definition must produce identical train/validation/test boundaries and eligible event counts for each matching $\alpha$. The code stops if these or the operating-policy parameters disagree. A foundation model uses only the fit portion of each training fold; the tail validation segment is reserved for probability-threshold selection.

In [ ]:
def load_fold_metrics(experiment_dir, source):
    frames = []
    for path in experiment_dir.glob('*/artifacts/cv/fold_metrics.csv'):
        run_dir = path.parents[2]
        with (run_dir / 'hparams.json').open(encoding='utf-8') as handle:
            config = json.load(handle)
        if float(config.get('target_threshold', -1)) != DEPEG_THRESHOLD_BPS:
            continue
        frame = pd.read_csv(path)
        frame['source'] = source
        frame['run_dir'] = str(run_dir)
        frame['source_mtime'] = path.stat().st_mtime
        frame['scaler'] = config.get('scaler')
        frame['target_window'] = config.get('target_window')
        frame['depeg_side'] = config.get('depeg_side')
        frame['dynamic_threshold'] = bool(config.get('dynamic_threshold', False))
        frame['false_alert_cost'] = config.get('false_alert_cost', 0.05)
        frame['min_lead_utility'] = config.get('min_lead_utility', 0.10)
        frame['utility_power'] = config.get('utility_power', 1.0)
        frame['alert_cooldown_hours'] = config.get('alert_cooldown_hours', 24.0)
        frame['min_lead_hours'] = config.get('min_lead_hours', config.get('warning_window_hours', [1, 24])[0])
        frame['max_lead_hours'] = config.get('max_lead_hours', config.get('warning_window_hours', [1, 24])[1])
        frame['utility_target_lead_hours'] = config.get('utility_target_lead_hours', frame['max_lead_hours'].iloc[0])
        frame['false_alert_budget_per_month'] = config.get('false_alert_budget_per_month')
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f'No fold metrics in {experiment_dir}')
    result = pd.concat(frames, ignore_index=True)
    result = result.loc[np.isclose(result['false_alert_budget_per_month'].astype(float), FALSE_ALERT_BUDGET)]
    return (result.sort_values('source_mtime')
                  .drop_duplicates(['source', 'model_name', 'alpha', 'fold'], keep='last')
                  .reset_index(drop=True))

folds = pd.concat([load_fold_metrics(TREE_DIR, 'Tree'),
                   load_fold_metrics(FOUNDATION_DIR, 'Foundation')], ignore_index=True)
folds = folds.loc[folds['alpha'].isin(summary['alpha'])].copy()
keys = ['alpha', 'fold']
audit_cols = ['train_size_full', 'train_size_fit', 'val_size', 'test_size',
              'train_end_timestamp', 'test_start', 'test_end', 'embargo_hours',
              'fold_n_events', 'scaler', 'target_window', 'depeg_side',
              'dynamic_threshold', 'false_alert_cost', 'min_lead_utility',
              'utility_power', 'alert_cooldown_hours', 'min_lead_hours',
              'max_lead_hours', 'utility_target_lead_hours',
              'false_alert_budget_per_month']
for field in audit_cols:
    disagreement = folds.groupby(['source', 'alpha', 'fold'])[field].nunique(dropna=False).gt(1)
    if disagreement.any():
        raise ValueError(f'Models within one experiment disagree on {field}.')
reference = folds.loc[folds['source'].eq('Tree')].drop_duplicates(keys)[keys + audit_cols]
foundation_folds = folds.loc[folds['source'].eq('Foundation')]
matched = foundation_folds.merge(reference, on=keys, how='left', suffixes=('_foundation', '_tree'), indicator=True)
if not matched['_merge'].eq('both').all():
    raise ValueError('Some foundation folds lack a matching tree fold at the same alpha.')
for field in audit_cols:
    left, right = matched[f'{field}_foundation'], matched[f'{field}_tree']
    if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
        okay = np.isclose(left.astype(float), right.astype(float), equal_nan=True).all()
    else:
        okay = left.astype(str).eq(right.astype(str)).all()
    if not okay:
        raise ValueError(f'CV mismatch in {field}; runs are not directly comparable.')
print(f'Comparability audit passed: {len(matched)} foundation model-fold rows match tree folds.')
display(folds.groupby(['source', 'model_name', 'alpha'])[['fold_n_events', 'fold_events_detected']]
        .agg(['count', 'min', 'max']).head(30))

## Operational comparison

Utility is primary; timely event recall, false alerts/month and median lead are deployment diagnostics. AUC, AUPRC, Brier score and Brier skill are secondary row-level metrics. Each interval is the run's event/calendar-block bootstrap 95% CI, averaged over its five overlapping outer folds.

In [ ]:
def show_ci(row, metric):
    value = row.get(f'cv_{metric}_mean')
    low = row.get(f'cv_{metric}_ci_lower')
    high = row.get(f'cv_{metric}_ci_upper')
    if pd.isna(value):
        return 'NA'
    if pd.isna(low) or pd.isna(high):
        return f'{value:.3f}'
    return f'{value:.3f} [{low:.3f}, {high:.3f}]'

ordered = summary.sort_values(['source', 'model_label', 'alpha']).copy()
comparison = pd.DataFrame({
    'Family': ordered['source'], 'Model': ordered['model_label'],
    'alpha': ordered['alpha'],
    'Event utility': ordered.apply(show_ci, axis=1, metric='event_utility_score'),
    'Timely recall': ordered.apply(show_ci, axis=1, metric='timely_event_recall'),
    'False alerts/month': ordered.apply(show_ci, axis=1, metric='false_alerts_per_month'),
    'Median lead (h)': ordered.apply(show_ci, axis=1, metric='median_lead_hours'),
    'AUC': ordered['cv_auc_mean'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
    'AUPRC': ordered['cv_auprc_mean'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
    'Brier': ordered['cv_brier_score_mean'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
    'Brier skill': ordered['cv_brier_skill_score_mean'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
})
display(comparison)
summary.to_csv(OUTPUT_DIR / 'foundation_vs_tree_cv_summary.csv', index=False)
def latex_escape(value):
    return str(value).replace('&', r'\&').replace('%', r'\%').replace('_', r'\_')

latex_lines = [
    r'\begin{table}[!htbp]', r'\centering\scriptsize',
    r'\caption{Operational 15-bp early-warning CV comparison at a validation false-alert budget of two episodes per month.}',
    r'\label{tab:foundation_vs_tree_cv}', r'\resizebox{\textwidth}{!}{%',
    r'\begin{tabular}{lllrrrrrrrr}', r'\toprule',
    ' & '.join(latex_escape(c) for c in comparison.columns) + r' \\',
    r'\midrule',
]
for row in comparison.itertuples(index=False, name=None):
    latex_lines.append(' & '.join(latex_escape(value) for value in row) + r' \\')
latex_lines += [r'\bottomrule', r'\end{tabular}}', r'\end{table}']
latex = '\n'.join(latex_lines) + '\n'
(OUTPUT_DIR / 'foundation_vs_tree_cv_table.tex').write_text(latex, encoding='utf-8')
print('Saved CSV and booktabs LaTeX table to', OUTPUT_DIR)

In [ ]:
metrics = [
    ('event_utility_score', 'Mean event utility'),
    ('timely_event_recall', 'Timely event recall'),
    ('false_alerts_per_month', 'False alerts/month'),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
for (metric, label), ax in zip(metrics, axes):
    for model, group in summary.groupby('model_label'):
        group = group.sort_values('alpha')
        y = group[f'cv_{metric}_mean'].to_numpy(dtype=float)
        lo = group[f'cv_{metric}_ci_lower'].to_numpy(dtype=float)
        hi = group[f'cv_{metric}_ci_upper'].to_numpy(dtype=float)
        lower = np.where(np.isfinite(lo), np.maximum(0, y - lo), 0)
        upper = np.where(np.isfinite(hi), np.maximum(0, hi - y), 0)
        ax.errorbar(group['alpha'], y, yerr=np.vstack([lower, upper]),
                    marker='o', linewidth=1.4, capsize=2.5, label=model)
    ax.set_title(label)
    ax.set_xlabel(r'$\alpha$')
    ax.grid(alpha=0.25)
axes[2].axhline(FALSE_ALERT_BUDGET, color='black', linestyle='--', linewidth=1, label='Validation budget')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.16), ncol=4, frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'foundation_vs_tree_operational_metrics.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'foundation_vs_tree_operational_metrics.pdf', bbox_inches='tight')
plt.show()

## Descriptive fold-paired difference

This last table pairs the leading tree and foundation model *within each* $\alpha$ using the same folds. It is descriptive only: the five test periods overlap, so a standard five-sample $t$-test would be invalid. A paired event/block resampling analysis of saved out-of-sample scores is needed for a formal superiority claim.

In [ ]:
leaders = (summary.sort_values('cv_event_utility_score_mean', ascending=False)
           .drop_duplicates(['source', 'alpha']))
chosen = folds.merge(leaders[['source', 'alpha', 'model_name']],
                     on=['source', 'alpha', 'model_name'], how='inner')
tree = chosen.loc[chosen['source'].eq('Tree'),
                  ['alpha', 'fold', 'model_name', 'fold_event_utility_score']].rename(
    columns={'model_name': 'tree_model', 'fold_event_utility_score': 'tree_utility'})
foundation = chosen.loc[chosen['source'].eq('Foundation'),
                        ['alpha', 'fold', 'model_name', 'fold_event_utility_score']].rename(
    columns={'model_name': 'foundation_model', 'fold_event_utility_score': 'foundation_utility'})
paired = tree.merge(foundation, on=['alpha', 'fold'], validate='one_to_one')
paired['foundation_minus_tree'] = paired['foundation_utility'] - paired['tree_utility']
display(paired.groupby(['alpha', 'tree_model', 'foundation_model'], as_index=False).agg(
    mean_difference=('foundation_minus_tree', 'mean'),
    min_difference=('foundation_minus_tree', 'min'),
    max_difference=('foundation_minus_tree', 'max'),
))
paired.to_csv(OUTPUT_DIR / 'foundation_minus_tree_paired_folds.csv', index=False)